In [ ]:
import pandas  as pd

In [ ]:


orders = pd.read_csv("olist_orders_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")

In [ ]:
customers

In [ ]:
customers["customer_id"].nunique()

In [ ]:
customers["customer_unique_id"].nunique()

In [ ]:
orders["order_id"].nunique()

In [ ]:
orders

In [ ]:
order_items.duplicated().sum()

In [ ]:
orders

In [ ]:
orders.info()

In [ ]:
orders.dtypes

#all the  orders columns are string first need to convert into date

In [ ]:
date_col= ["order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
    ]

for col in date_col:
    orders[col]= pd.to_datetime(orders[col],errors= 'coerce')
    

In [ ]:
orders.dtypes

In [ ]:
orders['order_status'].unique()

In [ ]:
orders['order_status'] = orders['order_status'].str.capitalize()

In [ ]:
orders['order_status'].unique()

In [ ]:
orders['order_status']= orders['order_status'].str.replace('Canceled','Cancelled')

In [ ]:
orders['order_status'].unique()

In [ ]:
orders.isnull().sum()

In [ ]:
# check order status

orders["order_status"].value_counts()

In [ ]:
orders[orders["order_status"]== "Shipped"]

"" duplicate  data""

In [ ]:
orders["order_id"].duplicated().sum()

In [ ]:
orders.head()

In [ ]:
#  order is  delivered  after purchase 


orders[(orders["order_purchase_timestamp"] > orders["order_delivered_customer_date"])]

In [ ]:
#  order is  approved  after purchase 

orders[(orders["order_purchase_timestamp"] > orders["order_approved_at"])]

In [ ]:
#  order delivered  after purchase 

orders[(orders["order_purchase_timestamp"] > orders["order_estimated_delivery_date"])]

In [ ]:
#  order delivered before   approved 

orders[(orders["order_approved_at"] > orders["order_delivered_customer_date"])]

In [ ]:
#   “I identified temporal inconsistencies where delivery occurred before approval (~0.06% of data). 
#    Since these records are logically invalid and negligible in volume, I excluded them to maintain data integrity.”

len(orders[orders["order_approved_at"] > orders["order_delivered_customer_date"]])* 100  / len(orders)

In [ ]:
#  get  only those data where the order is  approved  before  delivery 

orders =   orders [orders["order_delivered_customer_date"] >= orders["order_approved_at"] ]


In [ ]:
orders[orders['order_delivered_customer_date'] < orders['order_delivered_carrier_date']]

In [ ]:
# i indentify  some nconsitencies where  order is delivered before  carrier (~ .02 %)  which is to  small  in comparison to data 
# so  the best approach for this  to  remove  this   data 

len(orders[orders['order_delivered_customer_date'] < orders['order_delivered_carrier_date']]) * 100 / len(orders)

In [ ]:
# get only those orders where the  order is delivered  after carrier 

orders = orders[ orders["order_delivered_customer_date"] >= orders["order_delivered_carrier_date"] ] 

In [ ]:
orders[
    (orders['order_approved_at'].isnull()) & 
    (orders['order_delivered_customer_date'].notnull())
]

In [ ]:
orders[
    (orders['order_approved_at'].isnull()) & 
    (orders['order_purchase_timestamp'].notnull())
]

In [ ]:
#  calculate  delivery time  in days using dt.days  it  ignores  hours  miites and sec

orders["Delivery_time"]  = (orders["order_delivered_customer_date"] -  orders["order_purchase_timestamp"]) .dt.days

In [ ]:
orders

In [ ]:
# create  delay column  where the order  delivered  after  estimated  time 

orders["Delay"] = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.days

orders["Earliness"] = (orders["order_estimated_delivery_date"] - orders["order_delivered_customer_date"]).dt.days

In [ ]:
orders

In [ ]:
orders["Delivery_time"].describe()


In [ ]:
orders["Delay"].describe()

In [ ]:
#Order was delivered on the same day it was purchased

orders[orders["Delivery_time"]<= 0]

In [ ]:
orders["Delivery_Type"]  = orders["Delivery_time"].apply(
lambda x : "Same_Day"  if x ==  0 else "Standard"
    
)

In [ ]:
orders

In [ ]:
orders[orders["Delay"]> 0].shape

In [ ]:
orders.shape

In [ ]:
# “Although only 6.8% of orders are delayed, these delays may disproportionately impact customer satisfaction.”

len(orders[orders["Delay"] >0]) * 100 / len(orders)

In [ ]:
orders["Late"] = (orders["Delay"]> 0).astype(int)

In [ ]:
orders

In [ ]:
orders[orders["Late"]> 0]

In [ ]:
#=============================================================================
#“Do delayed deliveries lead to poor customer ratings?”
# REVIEW  CLEANING 

#====================================================================================

""REVIEW  CLEANING""

In [ ]:
Review = pd.read_csv("olist_order_reviews_dataset.csv")

In [ ]:
Review

In [ ]:
Review.info()

In [ ]:
Review["review_score"].unique()

In [ ]:
Review.isnull().sum()

In [ ]:
Review["order_id"].duplicated().sum()

In [ ]:
Review = Review.sort_values("review_answer_timestamp")

In [ ]:
Review

In [ ]:
#   keep only latest  finel  review 
# drop  duplicates  because may be  multiple  reviews  are gievn  from  same  order id

Review = Review.drop_duplicates(
    subset = "order_id",
    keep = "last"
)


In [ ]:
Review

In [ ]:
Review = Review[["order_id", "review_score"]]

In [ ]:
Review

In [ ]:
Review.isnull().sum()

In [ ]:
Review["order_id"].duplicated().sum()

In [ ]:
#==========================================================
#   CLEANING  DONE  OF  REVIEW  TABLE
#================================================

In [ ]:
#===============================================================
#     NEXT  STEP   MERGE   ORDERS  AND  REVIEW  TO  ANALYZE  CUSTOMER  SATISFACTION
#==============================================================

In [ ]:
#==========================================================
#  Customer_Saisfaction_score   as  CSS
#==========================================

In [ ]:
CSS  =  orders.merge(Review, on = "order_id", how = "inner")

In [ ]:
CSS

In [ ]:
Review.groupby("review_score")["review_score"].count()

In [ ]:
CSS[CSS["Late"]>0]

In [ ]:
#  ✅ SECTION 1: OVERALL CUSTOMER SATISFACTION

#Start simple → build context.

#        What is the overall average review score?
#        What is the distribution of review scores (1 to 5)?
#     What percentage of reviews are:
#     Bad (1–2)
#       Neutral (3)
#       Good (4–5)

In [ ]:
CSS["review_score"].mean()

In [ ]:
CSS.groupby("review_score")["review_score"].count()

In [ ]:
# Create category labels
CSS["review_category"] = pd.cut(
    CSS["review_score"],
    bins=[0, 2, 3, 5],
    labels=["Bad (1-2)", "Neutral (3)", "Good (4-5)"]
)

# Calculate percentage distribution
review_dist = CSS["review_category"].value_counts(normalize=True) * 100

# Sort for better readability
review_dist = review_dist.reindex(["Bad (1-2)", "Neutral (3)", "Good (4-5)"])

print(review_dist)

In [ ]:
CSS

In [ ]:
#  What is the average review score for:

#    Late deliveries
#    On-time/early deliveries


CSS.groupby("Late")["review_score"].mean()

In [ ]:
# What percentage of late deliveries result in bad reviews?

CSS [ (CSS["Late"]== 1)  & (CSS["review_category"] == "Bad (1-2)") ]

In [ ]:
CSS[CSS["Late"] ==1]["review_category"].value_counts(normalize = True) *100

In [ ]:
# What percentage of on-time deliveries result in bad reviews?

CSS[CSS["Late"] ==0]["review_category"].value_counts(normalize = True) *100


In [ ]:
CSS["is_bad_review"] = (CSS["review_score"] <= 2).astype(int)

In [ ]:
# Probability of bad review when late
p_late = CSS[CSS["Late"] == 1]["is_bad_review"].mean()

# Probability of bad review when not late
p_not_late = CSS[CSS["Late"] == 0]["is_bad_review"].mean()

# Increase in probability
increase = (p_late - p_not_late) * 100

print("P(Bad | Late):", round(p_late * 100, 2), "%")
print("P(Bad | Not Late):", round(p_not_late * 100, 2), "%")
print("Increase in probability:", round(increase, 2), "%")

In [ ]:
p_not_late

In [ ]:
CSS

In [ ]:
#   Do longer delays (e.g., 5+ days) lead to significantly worse ratings than small delays (1–2 days)?
import numpy as np
CSS["Delay_Buckets"] = pd.cut(
          CSS["Delay"],
           bins = [-np.inf, 0,2,5, np.inf],
           labels = ["on Time / Early ",  "(1-2)days", "(3-5) days" , "5+ days"], right= False
)

In [ ]:
CSS

In [ ]:
CSS.groupby("Delay_Buckets")["review_category"].value_counts(normalize = True) *100

In [ ]:
CSS.groupby("Delay_Buckets")["review_score"].mean()

In [ ]:
# Why?
#  Jump from 15% → 46% bad reviews
##  That’s a 3x increase
#   🚀 4. ANSWER TO YOUR ORIGINAL QUESTION

#  Do longer delays (5+ days) lead to significantly worse ratings than small delays (1–2 days)?

#  ✅ Answer:

#  Yes — significantly.

#  Avg rating drops from 3.91 → 1.78
#   Bad reviews jump from 15% → 77%


#   7. ACTIONABLE RECOMMENDATION

#“The company should prioritize reducing delivery delays beyond 3 days, as this is the threshold
# where customer dissatisfaction escalates significantly.”

In [ ]:
order_items = pd.read_csv("olist_order_items_dataset.csv")

In [ ]:
order_items[["order_id", "seller_id"]].duplicated().sum()

In [ ]:
order_items.duplicated().sum()

#“The original order_items dataset was stored at item-level granularity, causing repeated order–seller combinations
#when sellers sold multiple products within the same order. Since the analysis focused on seller performance and customer dissatisfaction, 
#the data was transformed into unique order–seller pairs by removing duplicate combinations of order_id and seller_id.”

In [ ]:
order_items = order_items[["order_id", "seller_id"]].drop_duplicates()

In [ ]:
order_items[["order_id", "seller_id"]].duplicated().sum()

In [ ]:
order_items.info()

In [ ]:
order_items.isnull().sum()

In [ ]:
order_items= order_items[["order_id","seller_id"]]

In [ ]:
order_items

In [ ]:
CSS = CSS.merge(order_items, on = "order_id", how= "left")

In [ ]:
CSS.shape

In [ ]:
CSS

In [ ]:
CSS.shape

In [ ]:
CSS.groupby("seller_id")["is_bad_review"].count()

In [ ]:
CSS["seller_id"].nunique()

In [ ]:
#  Which sellers have the highest average delivery delay?

# Late delay (only positive values, else 0)
CSS["late_delay"] = CSS["Delay"].clip(lower=0)

# Early delivery (only negative values → convert to positive magnitude)
CSS["early_days"] = (-CSS["Delay"]).clip(lower=0)

In [ ]:
seller_metrics = CSS.groupby("seller_id").agg(
    avg_delay=("Delay", "mean"),
    median_delay=("Delay", "median"),
    
    pct_late=("Delay", lambda x: (x > 0).mean() * 100),
    pct_early=("Delay", lambda x: (x < 0).mean() * 100),
    
    avg_late_delay=("late_delay", lambda x: x[x > 0].mean()),
    avg_early_days=("early_days", lambda x: x[x > 0].mean()),
    
    order_count=("order_id", "count")
).reset_index()

In [ ]:
seller_metrics["pct_late"] = seller_metrics["pct_late"].round(0).astype(int)
seller_metrics["pct_early"] = seller_metrics["pct_early"].round(0).astype(int)

In [ ]:
seller_metrics

In [ ]:
seller_metrics[seller_metrics["pct_late"] >50]

In [ ]:
seller_metrics[seller_metrics["pct_late"] >50]["order_count"].sum()

In [ ]:
#  high  risk  seller
high_risk_seller= seller_metrics[

    seller_metrics["pct_late"] >50 
]

In [ ]:
high_risk_seller

In [ ]:
TOTAL_ORDERS =  seller_metrics["order_count"].sum()

In [ ]:
TOTAL_ORDERS

In [ ]:
Order_by_high_risk_seller =   high_risk_seller["order_count"].sum()

In [ ]:
Total_risk_order_percent = (Order_by_high_risk_seller *100 / TOTAL_ORDERS)

In [ ]:
Total_risk_order_percent

In [ ]:
Order_by_high_risk_seller

In [ ]:
#  “Although a small group of sellers exhibits poor delivery performance, they contribute negligibly (<0.2%) to total order volume, 
#indicating that they are not the primary drivers of overall delays or customer dissatisfaction.”

In [ ]:
#   FIND SELLERS  WHO HAVE  HIGH ORDER VOLUME

seller_metrics = seller_metrics.sort_values("order_count", ascending = False)

In [ ]:
seller_metrics.head(20)

In [ ]:
seller_metrics["volume_segment"]= pd.qcut(

    seller_metrics["order_count"],
    q=3,
    labels= ["low","Medium","High"]
)

In [ ]:
seller_metrics

In [ ]:
seller_metrics["Delay_segments"] = pd.cut(

    seller_metrics["pct_late"],
    bins = [0,20,50,100],
    labels = ["low_delay", "Moderate_delay", "High_delay"]
)

In [ ]:
seller_metrics

In [ ]:
seller_metrics.groupby(["volume_segment","Delay_segments"])["order_count"].sum()

In [ ]:
# “While most orders are handled efficiently by high-volume sellers, a small portion of their deliveries still experience moderate delays. Low-volume sellers exhibit higher delay rates, 
# but their overall business impact is negligible.”



In [ ]:
seller_metrics.groupby(["volume_segment","Delay_segments"])["avg_late_delay"].mean()

In [ ]:
CSS = CSS.merge(
    seller_metrics[
        ["seller_id", "volume_segment", "Delay_segments"]
    ],
    on="seller_id",
    how="left"
)

In [ ]:
CSS.shape


In [ ]:
CSS

In [ ]:
CSS[["order_id","seller_id"]].nunique()

In [ ]:
CSS

In [ ]:
(CSS["is_bad_review"] ==1).sum() *100/ len(CSS)

In [ ]:
(CSS["is_bad_review"] ==1).mean() *100/ len(CSS)

In [ ]:
seller_metrics.groupby(["volume_segment","Delay_segments"])["order_count"].sum()

In [ ]:
# Customer dissatisfaction is driven more by moderate delays at scale than by extreme delays from small sellers.

In [ ]:
CSS[(CSS["volume_segment"] == "High") & (CSS["Delay_segments"]  == "Moderate_delay")]

In [ ]:
CSS.shape

In [ ]:
CSS.groupby(["volume_segment","Delay_segments"])["is_bad_review"].sum()

In [ ]:
#  Delivery delay influences DISsatisfaction, but large-scale dissatisfaction is driven primarily by high-volume sellers WITH LOW  DELAY , 
#  indicating additional operational or product-related issues beyond logistics delays.

In [ ]:
CSS.groupby(["volume_segment","Delay_segments"])["is_bad_review"].mean() *100

In [ ]:
#   FINAL INSIGHT 1
#Customer dissatisfaction rises sharply as delivery delays increase.

#✅ FINAL INSIGHT 2
#A critical operational threshold exists:
# beyond moderate delays,
  #bad review probability increases dramatically.

#✅ FINAL INSIGHT 3
# High-delay sellers have the worst operational performance, with dissatisfaction rates exceeding 60%.

#✅ FINAL INSIGHT 4
#  However, high-delay sellers contribute relatively little total business volume.

#✅ FINAL INSIGHT 5
#  Most total dissatisfaction comes from high-volume sellers because they process enormous order volumes, even though their operational performance is comparatively better.

In [ ]:
#Do delayed orders reduce repeat purchasing?

In [ ]:
CSS["order_id"].duplicated().sum()

In [ ]:
CSS["customer_id"].duplicated().sum()

In [ ]:
CSS.duplicated().sum()

In [ ]:
CSS[CSS.duplicated(subset = ["order_id", "customer_id"],
              keep = False)]

In [ ]:
#  CUSTOMER_REPETITION_ ANALYSIS  =  CRA

In [ ]:
CUSTOMER_FACT=  CSS

In [ ]:
CUSTOMER_FACT

In [ ]:
CUSTOMER_MATRICS = CUSTOMER_FACT.drop_duplicates(subset= ["order_id",  "customer_id"])

In [ ]:
CUSTOMER_MATRICS.duplicated(subset=["order_id", "customer_id"]).sum()

In [ ]:
CUSTOMER_MATRICS

In [ ]:
CUSTOMER_MATRICS.groupby("customer_id")["customer_id"].count()

In [ ]:
CUSTOMER_MATRICS["customer_id"].nunique()

In [ ]:
#2. CUSTOMER FACT TABLE ← NEW

#  Column Type	Examples
#  Order	order_id, timestamps
#  Customer	customer_id, customer_unique_id
#  Satisfaction	review_score, bad_review
# Delivery	delay, late
# Revenue	payment_value (optional)

In [ ]:
customers

In [ ]:
customers.info()

In [ ]:
customers.duplicated().sum()

In [ ]:
orders.info()

In [ ]:
orders

In [ ]:
CUSTOMER_FACT = orders.merge(customers, on = "customer_id" , how= "inner")

In [ ]:
CUSTOMER_FACT.info()

In [ ]:
CUSTOMER_FACT["customer_unique_id"].nunique()

In [ ]:
CUSTOMER_FACT.groupby("customer_unique_id")["order_id"].count().sort_values(ascending = False)

In [ ]:
payments

In [ ]:
payments.info()

In [ ]:
payments.duplicated().sum()

In [ ]:
payments["payment_sequential"].nunique()

In [ ]:
payments[payments["payment_installments"]<1]

In [ ]:
payments[payments["payment_value"]<0]

In [ ]:
payments[payments["order_id"].duplicated()]

In [ ]:
payments[payments["order_id"].duplicated(keep = False)].sort_values("order_id")

In [ ]:
payments["payment_type"] = payments["payment_type"].str.capitalize()

In [ ]:
payments["payment_type"].unique()

In [ ]:
payments

In [ ]:
sellers

In [ ]:
sellers.info()

In [ ]:
products

In [ ]:
products.info()

In [ ]:
products.duplicated().sum()

In [ ]:
products[products["product_name_lenght"] <0]

In [ ]:
products[products["product_description_lenght"] <0]

In [ ]:
products[products["product_photos_qty"] <0]

In [ ]:
products[products["product_weight_g"] <0]

In [ ]:
products[products["product_width_cm"] <0]

In [ ]:
products.isnull().sum()

In [ ]:
products["product_id"].unique()

In [ ]:
products= products.dropna()

In [ ]:
products.isnull().sum()

In [ ]:
from sqlalchemy import create_engine
import urllib

In [ ]:
import urllib
from sqlalchemy import create_engine

params = urllib.parse.quote_plus(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=NIF0825472\SQLEXPRESS;"
    r"DATABASE=Olist_Project;"
    r"Trusted_Connection=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}"
)

In [ ]:
engine.connect()

In [ ]:
Review.head(10).to_sql(
    "Review_Test",
    engine,
    if_exists="replace",
    index=False
)

In [ ]:

Review.to_sql("Review", engine, if_exists="replace", index=False)

payments.to_sql("Payments", engine, if_exists="replace", index=False)

sellers.to_sql("Sellers", engine, if_exists="replace", index=False)

products.to_sql("Products", engine, if_exists="replace", index=False)

orders.to_sql("Orders", engine, if_exists="replace", index=False)

customers.to_sql("Customers", engine, if_exists="replace", index=False)

CUSTOMER_MATRICS.to_sql("Customer_Matrics", engine, if_exists="replace", index=False)

CSS.to_sql("Customer_Satisfaction_Score", engine, if_exists="replace", index=False)

seller_metrics.to_sql("Seller_Metrics", engine, if_exists="replace", index=False)

In [ ]:
order_items.to_sql("order_items", engine, if_exists="replace", index=False)


In [ ]:
CUSTOMER_FACT

In [ ]:
payments.duplicated().sum()

In [ ]:
payments.isnull().sum()

In [ ]:
CUSTOMER_FACT.dtypes

In [ ]:
datetime_cols = CUSTOMER_FACT.select_dtypes(include=["datetime64"]).columns

for col in datetime_cols:
    CUSTOMER_FACT[col] = pd.to_datetime(CUSTOMER_FACT[col])

In [ ]:
CSS["customer_unique_id"] = CUSTOMER_FACT["customer_unique_id"]

In [ ]:
CSS = CSS.drop("customer_unique_id", axis=1)

In [ ]:
CSS = CSS.merge(
    CUSTOMER_FACT[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

In [ ]:
CSS


In [ ]:
CSS.to_sql("Customer_Satisfaction_Score", engine, if_exists="replace", index=False)
